In [1]:
import pandas as pd

emp = pd.read_csv("employees_raw.csv")

print(emp.shape)
print(emp.dtypes)
emp.head(10)

(7, 5)
emp_id          int64
emp_name          str
dept_id       float64
manager_id    float64
salary          int64
dtype: object


,emp_id,emp_name,dept_id,manager_id,salary
0,1,Asha,10.0,NaN,120000
1,2,Ravi,10.0,1.0,80000
2,3,Meena,20.0,1.0,75000
3,4,Karan,20.0,3.0,60000
4,5,Divya,NaN,1.0,70000
5,6,Arjun,50.0,3.0,55000
6,4,Karan,20.0,3.0,60000


In [2]:
emp["dept_id"] = emp["dept_id"].astype("Int64")

In [4]:
emp.duplicated().sum()

np.int64(1)

In [5]:
emp.isna().sum()

emp_id        0
emp_name      0
dept_id       1
manager_id    1
salary        0
dtype: int64

In [6]:
print("Duplicates:", emp.duplicated().sum())
print(emp.isna().sum())

emp = emp.drop_duplicates()
emp["dept_id"] = emp["dept_id"].astype("Int64")
emp["manager_id"] = emp["manager_id"].astype("Int64")

print(emp.shape)
print(emp.dtypes)
emp

Duplicates: 1
emp_id        0
emp_name      0
dept_id       1
manager_id    1
salary        0
dtype: int64
(6, 5)
emp_id        int64
emp_name        str
dept_id       Int64
manager_id    Int64
salary        int64
dtype: object


,emp_id,emp_name,dept_id,manager_id,salary
0,1,Asha,10,<NA>,120000
1,2,Ravi,10,1,80000
2,3,Meena,20,1,75000
3,4,Karan,20,3,60000
4,5,Divya,<NA>,1,70000
5,6,Arjun,50,3,55000


In [7]:
   emp["salary"] > 70000      # True/False for each of the 6 rows

0     True
1     True
2     True
3    False
4    False
5    False
Name: salary, dtype: bool

In [8]:
   emp[emp["salary"] > 70000]

,emp_id,emp_name,dept_id,manager_id,salary
0,1,Asha,10,<NA>,120000
1,2,Ravi,10,1,80000
2,3,Meena,20,1,75000


In [9]:
emp.loc[(emp["dept_id"] == 10) & (emp["salary"] > 70000), ["emp_name", "salary"]]

,emp_name,salary
0,Asha,120000
1,Ravi,80000


#### F1. Employees in dept 10 or dept 20, written with isin.

In [16]:
emp[emp['dept_id'].isin([10,20])]

,emp_id,emp_name,dept_id,manager_id,salary
0,1,Asha,10,<NA>,120000
1,2,Ravi,10,1,80000
2,3,Meena,20,1,75000
3,4,Karan,20,3,60000


#### F2. Employees with no manager or earning less than 60,000.

In [11]:
emp[(emp['manager_id'].isna()) | (emp['salary'] < 60000)]

,emp_id,emp_name,dept_id,manager_id,salary
0,1,Asha,10,<NA>,120000
5,6,Arjun,50,3,55000


#### F3. Employees who are not in dept 10, written with ~. Careful: think about whether Divya (dept <NA>) is included. In SQL, WHERE dept_id <> 10 would drop her. Does Pandas behave the same way?

In [17]:
emp[~(emp['dept_id'].isin([10]))]

,emp_id,emp_name,dept_id,manager_id,salary
2,3,Meena,20,1,75000
3,4,Karan,20,3,60000
4,5,Divya,<NA>,1,70000
5,6,Arjun,50,3,55000


In [19]:
emp[~(emp['dept_id']== 10)]

,emp_id,emp_name,dept_id,manager_id,salary
2,3,Meena,20,1,75000
3,4,Karan,20,3,60000
5,6,Arjun,50,3,55000


#### G1. Run the example below with dropna=False. How many rows come back, and what is Analytics’ (dept 10) avg_salary? Then run it without dropna=False. Which row goes missing?

In [24]:
emp.groupby("dept_id", dropna=False).agg(
    headcount=("emp_id", "count"),
    avg_salary=("salary", "mean"),
).reset_index()

,dept_id,headcount,avg_salary
0,10,2,100000.0
1,20,2,67500.0
2,50,1,55000.0
3,<NA>,1,70000.0


#### G2. For each dept_id, give the headcount, the total salary and the highest salary. Keep only departments with headcount ≥ 2. That’s HAVING, and in Pandas you filter the result after agg(), the same way as in Step 3.

In [26]:
emp.groupby("dept_id", dropna=False).agg(
    headcount=("emp_id", "count"),
    total_salary=("salary", "sum"),
    highest_salary= ("salary", "max")
).reset_index().query("headcount >= 2")

,dept_id,headcount,total_salary,highest_salary
0,10,2,200000,120000
1,20,2,135000,75000


In [27]:
dept = pd.read_csv("departments.csv")

emp.merge(dept, on="dept_id", how="inner", indicator=True)

,emp_id,emp_name,dept_id,manager_id,salary,dept_name,_merge
0,1,Asha,10,<NA>,120000,Analytics,both
1,2,Ravi,10,1,80000,Analytics,both
2,3,Meena,20,1,75000,Finance,both
3,4,Karan,20,3,60000,Finance,both


#### M1. emp INNER JOIN dept: how many rows, and who drops out?

In [28]:
emp.merge(dept, on="dept_id", how="inner", indicator=True)

,emp_id,emp_name,dept_id,manager_id,salary,dept_name,_merge
0,1,Asha,10,<NA>,120000,Analytics,both
1,2,Ravi,10,1,80000,Analytics,both
2,3,Meena,20,1,75000,Finance,both
3,4,Karan,20,3,60000,Finance,both


#### M2. emp LEFT JOIN dept: how many rows, and which employees get a blank dept_name?

In [29]:
emp.merge(dept, on="dept_id", how="left", indicator=True)

,emp_id,emp_name,dept_id,manager_id,salary,dept_name,_merge
0,1,Asha,10,<NA>,120000,Analytics,both
1,2,Ravi,10,1,80000,Analytics,both
2,3,Meena,20,1,75000,Finance,both
3,4,Karan,20,3,60000,Finance,both
4,5,Divya,<NA>,1,70000,NaN,left_only
5,6,Arjun,50,3,55000,NaN,left_only


#### M3. FULL OUTER JOIN: how many rows in total?

In [30]:
emp.merge(dept, on="dept_id", how="outer", indicator=True)

,emp_id,emp_name,dept_id,manager_id,salary,dept_name,_merge
0,1.0,Asha,10,<NA>,120000.0,Analytics,both
1,2.0,Ravi,10,1,80000.0,Analytics,both
2,3.0,Meena,20,1,75000.0,Finance,both
3,4.0,Karan,20,3,60000.0,Finance,both
4,NaN,NaN,30,<NA>,NaN,Marketing,right_only
5,NaN,NaN,40,<NA>,NaN,HR,right_only
6,6.0,Arjun,50,3,55000.0,NaN,left_only
7,5.0,Divya,<NA>,1,70000.0,NaN,left_only


#### M4. An anti-join in the other direction: departments that have no employees. Hint: put dept on the left.

In [31]:
m = dept.merge(emp, on="dept_id", how="left", indicator=True)
m[m["_merge"] == "left_only"]

,dept_id,dept_name,emp_id,emp_name,manager_id,salary,_merge
4,30,Marketing,NaN,NaN,<NA>,NaN,left_only
5,40,HR,NaN,NaN,<NA>,NaN,left_only
